# Preparação do Ambiente

# Análise por País

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import plotly.express as px
import plotly.graph_objects as go

In [ ]:
import pandas as pd


df = pd.read_csv("../../dataset/Combined_dataset.csv")



,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Canada,SE649035-145565,River,12-01-1974,0.059248,1.30,8.1500,0.011917,8.07500,9.885,0.343917,11.73155,100.0,Excellent
1,Canada,SE649035-145565,River,12-01-1975,0.039821,1.38,7.8000,0.009417,7.73333,10.150,0.449083,11.82009,100.0,Excellent
2,Canada,SE649035-145565,River,12-01-1976,0.031341,2.23,7.8000,0.011000,7.46667,10.235,0.220750,14.87472,100.0,Excellent
3,Canada,SE649035-145565,River,12-01-1977,0.020501,1.61,8.1500,0.012333,7.78333,11.116,0.572250,15.89293,100.0,Excellent
4,Canada,SE649035-145565,River,12-01-1978,0.020023,1.64,4.3708,0.006182,7.10000,7.068,0.371091,15.22888,100.0,Excellent


# Análise por País

## Contagem de amostras por país
Agora vamos analisar a quantidade de amostras por país para compreender a representatividade de cada localidade no dataset e identificar possíveis desbalanceamentos na distribuição geográfica dos dados.


---
**Pergunta que queremos responder**


*   A distribuição de amostras entre os países é equilibrada ou há desbalanceamento na representatividade dos dados?


---

**Objetivo**


*   Identificar a quantidade de registros por país, avaliando a distribuição geográfica do dataset e possíveis concentrações que possam gerar vieses nas análises posteriores






In [ ]:
if 'Country' in df.columns:
    country_counts = df['Country'].value_counts()
    country_percentage = df['Country'].value_counts(normalize=True) * 100

    country_summary = pd.DataFrame({
        'Contagem': country_counts,
        'Percentual (%)': country_percentage
    })

    print('Distribuição de dados por País:')
    display(country_summary)

Distribuição de dados por País:


,Contagem,Percentual (%)
Country,,
England,2129198,75.290499
USA,413814,14.632863
Ireland,235019,8.310499
China,45997,1.626498
Canada,3949,0.139640


A contagem de dados por país revela uma distribuição desigual das amostras no dataset. A Inglaterra possui a maioria dos registros, seguida pelos Estados Unidos e pela Irlanda. A China e o Canadá apresentam o menor número de registros, com o Canadá sendo o país com menos amostras. Essa distribuição pode afetar a generalização dos resultados, e as análises futuras devem considerar o viés de representatividade entre os países.

## Intervalo temporal por país
Agora vamos identificar o ano mínimo e o ano máximo em que há registros para cada país, a fim de obter uma visão geral da cobertura temporal do dataset.


---

**Pergunta que queremos responder**


*   Qual é o intervalo temporal dos registros disponíveis para cada país?



---

**Objetivo**


*   Compreender a cobertura temporal dos dados por país, identificando o período inicial e final de registros disponível para cada localidade

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y', errors='coerce')
df['Year'] = df['Date'].dt.year

if 'Year' in df.columns and 'Country' in df.columns:
    country_year_range = df.groupby('Country').agg(
        Ano_Inicial=('Year', 'min'),
        Ano_Final=('Year', 'max')
    )
    print('Intervalo de Anos por País:')
    display(country_year_range)

Intervalo de Anos por País:


,Ano_Inicial,Ano_Final
Country,,
Canada,1968,2021
China,2001,2017
England,2000,2023
Ireland,1940,2023
USA,1940,2023


A análise do intervalo de anos por país revela diferenças significativas na cobertura temporal dos dados. A Irlanda e os Estados Unidos apresentam o maior intervalo, de 1940 a 2023, indicando uma longa amplitude temporal de registros. A Inglaterra também possui uma cobertura ampla, de 2000 a 2023. Já o Canadá apresenta dados entre 1968 e 2021, enquanto a China possui um período mais recente e reduzido, de 2001 a 2017.

Essa variação na cobertura temporal é importante para análises de séries históricas e comparações entre países, pois pode influenciar a representatividade das tendências ao longo do tempo. Vale destacar que o intervalo apresentado refere-se apenas ao período entre o primeiro e o último registro, não garantindo necessariamente continuidade nos dados ao longo dos anos.

## Análise de frequência anual e lacunas temporais por país
Vamos investigar a frequência de registros por ano dentro do intervalo total de dados de cada país para identificar possíveis lacunas ou períodos de baixa amostragem.

---

**Pergunta que queremos responder**

*   Existem anos sem registros ou com poucos registros entre o ano mínimo e máximo de cada país?

---

**Objetivo**

*   Identificar lacunas temporais significativas que podem afetar análises de séries temporais.
*   Entender a consistência da coleta de dados ao longo do tempo para cada localidade.

In [ ]:
yearly_counts = df.groupby(['Country', 'Year']).size().reset_index(name='count')

print("Análise de Lacunas Temporais por País:")

for country in df['Country'].unique():
    country_data = yearly_counts[yearly_counts['Country'] == country]

    if not country_data.empty:
        min_year = country_data['Year'].min()
        max_year = country_data['Year'].max()

        all_years_in_range = set(range(int(min_year), int(max_year) + 1))
        present_years = set(country_data['Year'].astype(int))

        missing_years = sorted(list(all_years_in_range - present_years))

        print(f"\nPaís: {country}")
        print(f"  Intervalo de Anos com Dados: {int(min_year)} - {int(max_year)}")

        if missing_years:
            print(f"  Anos sem registros dentro do intervalo: {missing_years}")
        else:
            print("  Nenhum ano sem registros dentro do intervalo (cobertura contínua).")

        # Optional: identify years with very low counts (e.g., less than 10 records)
        low_data_years = country_data[country_data['count'] < 10] # Threshold can be adjusted
        if not low_data_years.empty:
            print(f"  Anos com poucos registros (menos de 10): {list(zip(low_data_years['Year'].astype(int), low_data_years['count']))}")
        else:
            print("  Nenhum ano com poucos registros (menos de 10) dentro do intervalo.")

    else:
        print(f"\nPaís: {country} - Sem dados anuais.")

Análise de Lacunas Temporais por País:

País: Canada
  Intervalo de Anos com Dados: 1968 - 2021
  Nenhum ano sem registros dentro do intervalo (cobertura contínua).
  Anos com poucos registros (menos de 10): [(1968, 1), (1969, 4), (1970, 3), (1971, 6), (1972, 6), (1973, 6), (1974, 7), (1975, 9), (2014, 9), (2015, 6), (2016, 6), (2017, 7)]

País: England
  Intervalo de Anos com Dados: 2000 - 2023
  Nenhum ano sem registros dentro do intervalo (cobertura contínua).
  Nenhum ano com poucos registros (menos de 10) dentro do intervalo.

País: China
  Intervalo de Anos com Dados: 2001 - 2017
  Nenhum ano sem registros dentro do intervalo (cobertura contínua).
  Nenhum ano com poucos registros (menos de 10) dentro do intervalo.

País: USA
  Intervalo de Anos com Dados: 1940 - 2023
  Nenhum ano sem registros dentro do intervalo (cobertura contínua).
  Nenhum ano com poucos registros (menos de 10) dentro do intervalo.

País: Ireland
  Intervalo de Anos com Dados: 1940 - 2023
  Anos sem registro

A análise da frequência anual e das lacunas temporais por país revela diferenças importantes na consistência da coleta de dados ao longo do tempo. Inglaterra, China e USA apresentam cobertura contínua dentro de seus respectivos intervalos, sem anos ausentes e sem anos com menos de 10 registros, indicando uma distribuição temporal mais regular e favorável para análises de séries temporais.

O Canadá, embora também apresente cobertura contínua, possui vários anos com baixa quantidade de registros, especialmente no início e no final do período analisado. Isso sugere que, apesar da ausência de lacunas completas, alguns anos podem ter amostragem insuficiente para análises temporais mais detalhadas.

A Irlanda se destaca por apresentar o maior número de lacunas temporais. Apesar de possuir um intervalo amplo de 1940 a 2023, muitos anos dentro desse período não possuem registros, e alguns dos anos com dados apresentam baixa amostragem. Essa irregularidade pode comprometer análises de tendências de longo prazo e comparações históricas, exigindo maior cautela na interpretação.

Esses resultados mostram que a amplitude temporal, por si só, não garante consistência na coleta. Por isso, a qualidade temporal dos dados deve ser considerada na escolha dos países mais adequados para análises de séries temporais.

## Desbalanceamento por país
Agora vamos analisar o desbalanceamento entre os países para verificar se a distribuição das amostras é equilibrada ou se há predominância de determinadas localidades no dataset.


---
**Pergunta que queremos responder**


*  A distribuição de amostras entre os países é balanceada ou existe predominância de alguns países no dataset?


---

**Objetivo**


*   identificar se há desbalanceamento na representatividade dos países, avaliando o impacto dessa concentração nas análises posteriores e no potencial viés dos resultados







In [ ]:
if 'Country' in df.columns:
    country_counts = df['Country'].value_counts()
    country_percentage = df['Country'].value_counts(normalize=True) * 100

    country_balance = pd.DataFrame({
        'Contagem': country_counts,
        'Percentual (%)': country_percentage.round(2)
    })

    print('Desbalanceamento de Amostras por País:')
    display(country_balance)
else:
    print("A coluna 'Country' não foi encontrada no DataFrame.")

Desbalanceamento de Amostras por País:


,Contagem,Percentual (%)
Country,,
England,2129198,75.29
USA,413814,14.63
Ireland,235019,8.31
China,45997,1.63
Canada,3949,0.14


A análise do desbalanceamento por país revela uma distribuição desigual das amostras, com forte predominância da Inglaterra (≈75%), seguida por Estados Unidos (≈14%) e Irlanda (≈8%). Já China e Canadá apresentam participação bastante reduzida.

Esse cenário indica que o dataset não é equilibrado entre os países, o que pode introduzir viés nas análises, fazendo com que os resultados reflitam principalmente os padrões dos países mais representados.